In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression # Model Klasifikasi Terbaik
from sklearn.metrics import classification_report
import joblib # Untuk menyimpan model

# 1. Muat Dataset Anda
file_path = 'DATASET_SENTIMEN_GENERATED_FINAL.csv'
df = pd.read_csv(file_path)

# 2. Bersihkan Data (Pastikan tidak ada NaN di teks atau label)
df.dropna(subset=['DATA', 'LABEL'], inplace=True)
df['DATA'] = df['DATA'].astype(str) # Pastikan kolom teks bertipe string

# 3. Definisikan Fitur (X) dan Target (y)
X = df['DATA']
y = df['LABEL']

print(f"Total data bersih untuk training: {len(df)} baris.")

Total data bersih untuk training: 11777 baris.


In [2]:
# 1. Bagi data menjadi Training dan Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Vektorisasi Teks
vectorizer = TfidfVectorizer(max_features=5000) # Batasi 5000 fitur teratas
X_train_vectorized = vectorizer.fit_transform(X_train)

# 3. Latih Model (Logistic Regression)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vectorized, y_train)

# 4. Evaluasi Kinerja Model
X_test_vectorized = vectorizer.transform(X_test)
y_pred = model.predict(X_test_vectorized)

print("\n## 📊 Kinerja Model Klasifikasi (Logistic Regression):")
print(classification_report(y_test, y_pred))


## 📊 Kinerja Model Klasifikasi (Logistic Regression):
              precision    recall  f1-score   support

     NEGATIF       0.94      0.94      0.94       792
      NETRAL       0.92      0.91      0.92       735
     POSITIF       0.95      0.96      0.96       829

    accuracy                           0.94      2356
   macro avg       0.94      0.94      0.94      2356
weighted avg       0.94      0.94      0.94      2356



In [3]:
# 1. Simpan Model Klasifikasi
model_filename = 'sentimen_model_lr.pkl'
joblib.dump(model, model_filename)

# 2. Simpan Vectorizer (PENTING: Harus disimpan bersama model)
vectorizer_filename = 'tfidf_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_filename)

print(f"\n✅ Model berhasil disimpan sebagai: {model_filename}")
print(f"✅ Vectorizer berhasil disimpan sebagai: {vectorizer_filename}")


✅ Model berhasil disimpan sebagai: sentimen_model_lr.pkl
✅ Vectorizer berhasil disimpan sebagai: tfidf_vectorizer.pkl


# AUTOMATED LABELING

In [1]:
import pandas as pd
import joblib # Untuk memuat model dan vectorizer
import os

# === KONFIGURASI FILE ===
FILE_TANPA_LABEL = 'data_tanpa_label.csv'
MODEL_FILENAME = 'sentimen_model_lr.pkl'
VECTORIZER_FILENAME = 'tfidf_vectorizer.pkl'
OUTPUT_FILENAME = 'data_TERLABEL_OTOMATIS.csv'
TEXT_COLUMN = 'DATA' # <--- PASTIKAN NAMA KOLOM TEKS ANDA SAMA DENGAN INI

# --- 1. MEMUAT MODEL DAN DATA ---
try:
    # Muat kembali model dan vectorizer
    loaded_model = joblib.load(MODEL_FILENAME)
    loaded_vectorizer = joblib.load(VECTORIZER_FILENAME)

    # Muat data yang belum berlabel
    df_unlabeled = pd.read_csv(FILE_TANPA_LABEL)

    print("✅ Model, Vectorizer, dan Data berhasil dimuat.")

except FileNotFoundError as e:
    print(f"❌ ERROR: File tidak ditemukan: {e}")
    print("Pastikan semua file (CSV dan .pkl) berada di direktori yang sama.")
    exit()

# --- 2. PERSIAPAN DATA ---
if TEXT_COLUMN not in df_unlabeled.columns:
    print(f"❌ ERROR: Kolom '{TEXT_COLUMN}' tidak ditemukan di {FILE_TANPA_LABEL}.")
    print("Mohon ganti variabel TEXT_COLUMN dengan nama kolom teks yang benar.")
    exit()

# Bersihkan dan pastikan kolom teks adalah string
df_unlabeled[TEXT_COLUMN] = df_unlabeled[TEXT_COLUMN].fillna('').astype(str)

# --- 3. PELABELAN OTOMATIS (PREDIKSI) ---

# Ambil teks yang akan diprediksi
X_unlabeled = df_unlabeled[TEXT_COLUMN]

# 1. Vektorisasi teks baru (PENTING: Hanya TRANSFORM, jangan FIT)
X_unlabeled_vectorized = loaded_vectorizer.transform(X_unlabeled)

# 2. Prediksi Label
predictions = loaded_model.predict(X_unlabeled_vectorized)

# 3. Masukkan hasil prediksi ke kolom 'LABEL'
df_unlabeled['LABEL'] = predictions

print(f"\n✅ Prediksi sentimen pada {len(df_unlabeled)} baris selesai.")

# --- 4. PENYIMPANAN HASIL ---

# Simpan DataFrame yang sudah terisi label ke file CSV baru
df_unlabeled.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')

print("\n--- HASIL AKHIR ---")
print(f"File '{FILE_TANPA_LABEL}' berhasil dilabeli secara otomatis.")
print(f"Data final disimpan sebagai: {OUTPUT_FILENAME}")
print(f"Distribusi Label Hasil Prediksi:")
print(df_unlabeled['LABEL'].value_counts())

✅ Model, Vectorizer, dan Data berhasil dimuat.

✅ Prediksi sentimen pada 37107 baris selesai.

--- HASIL AKHIR ---
File 'data_tanpa_label.csv' berhasil dilabeli secara otomatis.
Data final disimpan sebagai: data_TERLABEL_OTOMATIS.csv
Distribusi Label Hasil Prediksi:
LABEL
NETRAL     16619
POSITIF    12471
NEGATIF     8017
Name: count, dtype: int64
